In [1]:
import pandas as pd
from pathlib import Path

from modules.evaluation import (
    calculate_mae,
    calculate_rmse,
    calculate_coverage,
    calculate_precision_at_n,
    calculate_recall_at_n,
    calculate_access_decision_consistency
)

DATA_DIR = Path("data")
RESULTS_DIR = Path("results")
RESULTS_DIR.mkdir(exist_ok=True)

rt_clean = pd.read_csv(DATA_DIR / "rt_clean.csv")

final_trust_df = pd.read_csv(RESULTS_DIR / "final_trust_user_0_sample.csv")
recommendations_df = pd.read_csv(RESULTS_DIR / "top_n_recommendations_user_0.csv")
detection_metrics_df = pd.read_csv(RESULTS_DIR / "detection_metrics.csv")
access_df = pd.read_csv(RESULTS_DIR / "access_decisions_user_0.csv")

print("Clean RT shape:", rt_clean.shape)
print("Final trust shape:", final_trust_df.shape)
print("Recommendations shape:", recommendations_df.shape)
print("Detection metrics shape:", detection_metrics_df.shape)
print("Access decisions shape:", access_df.shape)

Clean RT shape: (339, 5825)
Final trust shape: (20, 6)
Recommendations shape: (5, 7)
Detection metrics shape: (6, 2)
Access decisions shape: (5, 6)


In [2]:
target_user = 0

actual_scores = []
predicted_scores = []

for _, row in final_trust_df.iterrows():
    service_id = int(row["service_id"])

    actual_score = rt_clean.iloc[target_user, service_id]
    predicted_score = row["final_trust"]

    actual_scores.append(actual_score)
    predicted_scores.append(predicted_score)

mae = calculate_mae(actual_scores, predicted_scores)
rmse = calculate_rmse(actual_scores, predicted_scores)

print("MAE:", mae)
print("RMSE:", rmse)

MAE: 0.06156833638251417
RMSE: 0.0862612509786434


In [3]:
coverage = calculate_coverage(
    prediction_df=final_trust_df,
    score_column="final_trust"
)

print("Coverage:", coverage)
print("Coverage Percentage:", coverage * 100)

Coverage: 1.0
Coverage Percentage: 100.0


In [4]:
candidate_services = final_trust_df["service_id"].astype(int).tolist()

precision_at_n = calculate_precision_at_n(
    recommendations_df=recommendations_df,
    actual_quality_df=rt_clean,
    target_user=target_user,
    relevance_threshold=0.60
)

recall_at_n = calculate_recall_at_n(
    recommendations_df=recommendations_df,
    actual_quality_df=rt_clean,
    candidate_services=candidate_services,
    target_user=target_user,
    relevance_threshold=0.60
)

print("Precision@N:", precision_at_n)
print("Recall@N:", recall_at_n)

Precision@N: 1.0
Recall@N: 0.25


In [5]:
access_consistency = calculate_access_decision_consistency(access_df)

print("Access Decision Consistency:", access_consistency)
print("Access Decision Consistency Percentage:", access_consistency * 100)

Access Decision Consistency: 1.0
Access Decision Consistency Percentage: 100.0


In [6]:
detection_metrics_df

,metric,value
0,Known Malicious Users,34.0
1,Extreme Deviation Flagged Users,19.0
2,CUSUM Flagged Users,34.0
3,Total Combined Flagged Users,34.0
4,Detection Rate,1.0
5,False Positive Rate,0.0


In [7]:
final_evaluation_df = pd.DataFrame({
    "Metric": [
        "MAE",
        "RMSE",
        "Coverage",
        "Precision@N",
        "Recall@N",
        "Detection Rate",
        "False Positive Rate",
        "Access Decision Consistency"
    ],
    "Value": [
        mae,
        rmse,
        coverage,
        precision_at_n,
        recall_at_n,
        float(detection_metrics_df.loc[detection_metrics_df["metric"] == "Detection Rate", "value"].iloc[0]),
        float(detection_metrics_df.loc[detection_metrics_df["metric"] == "False Positive Rate", "value"].iloc[0]),
        access_consistency
    ]
})

final_evaluation_df.to_csv(
    RESULTS_DIR / "final_evaluation_metrics.csv",
    index=False
)

final_evaluation_df

,Metric,Value
0,MAE,0.061568
1,RMSE,0.086261
2,Coverage,1.000000
3,Precision@N,1.000000
4,Recall@N,0.250000
5,Detection Rate,1.000000
6,False Positive Rate,0.000000
7,Access Decision Consistency,1.000000


In [8]:
summary_rows = []

summary_rows.append({
    "Result Area": "Data Preprocessing",
    "Output": "rt_clean.csv and rt_malicious.csv created",
    "Interpretation": "WS-DREAM response-time data cleaned, transformed to quality score, and malicious users injected."
})

summary_rows.append({
    "Result Area": "Direct Trust",
    "Output": "direct_trust_user_0_sample.csv",
    "Interpretation": "PCC, Cosine Similarity, AOI filtering, and Top K neighbour selection were implemented."
})

summary_rows.append({
    "Result Area": "Indirect Trust",
    "Output": "indirect_trust_user_0_sample.csv",
    "Interpretation": "BFS-based trust diffusion with depth decay was implemented."
})

summary_rows.append({
    "Result Area": "Weighted Aggregation",
    "Output": "final_trust_user_0_sample.csv",
    "Interpretation": "Direct and indirect trust were combined using alpha = 0.6 and beta = 0.4."
})

summary_rows.append({
    "Result Area": "Anomaly Detection",
    "Output": "flagged_users.csv and detection_metrics.csv",
    "Interpretation": "Injected malicious users were detected using deviation and CUSUM-based checks."
})

summary_rows.append({
    "Result Area": "Top N Recommendation",
    "Output": "top_n_recommendations_user_0.csv",
    "Interpretation": "Top trusted services above the 0.60 trust threshold were recommended."
})

summary_rows.append({
    "Result Area": "RBAC Access Decision",
    "Output": "access_decisions_user_0.csv",
    "Interpretation": "Recommended services were checked against RBAC policy before access was granted or denied."
})

summary_df = pd.DataFrame(summary_rows)

summary_df.to_csv(
    RESULTS_DIR / "final_result_summary.csv",
    index=False
)

summary_df

,Result Area,Output,Interpretation
0,Data Preprocessing,rt_clean.csv and rt_malicious.csv created,"WS-DREAM response-time data cleaned, transform..."
1,Direct Trust,direct_trust_user_0_sample.csv,"PCC, Cosine Similarity, AOI filtering, and Top..."
2,Indirect Trust,indirect_trust_user_0_sample.csv,BFS-based trust diffusion with depth decay was...
3,Weighted Aggregation,final_trust_user_0_sample.csv,Direct and indirect trust were combined using ...
4,Anomaly Detection,flagged_users.csv and detection_metrics.csv,Injected malicious users were detected using d...
5,Top N Recommendation,top_n_recommendations_user_0.csv,Top trusted services above the 0.60 trust thre...
6,RBAC Access Decision,access_decisions_user_0.csv,Recommended services were checked against RBAC...
